# Tarefa de casa - Estruturar dataset da Aula 4

Rodrigo Gimenes Rodrigues - CP3071782

Pipeline: outliers (IQR em total_stats), scaling numérico e encoding dos tipos.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

df = pd.read_csv('../Aula5_Limpeza_e_Tratamento/pokemon_limpo.csv')
print(df.shape)
df.head()

In [ ]:
# 1) outliers em total_stats (IQR) - remoção
q1 = df['total_stats'].quantile(0.25)
q3 = df['total_stats'].quantile(0.75)
iqr = q3 - q1
li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('limites total_stats:', round(li,1), round(ls,1))
print('outliers:', ((df['total_stats'] < li) | (df['total_stats'] > ls)).sum())
df2 = df[(df['total_stats'] >= li) & (df['total_stats'] <= ls)].copy()

# 2) scaling de colunas numéricas
num_cols = ['altura_m', 'peso_kg', 'experiencia_base', 'hp', 'ataque', 'defesa',
            'ataque_especial', 'defesa_especial', 'velocidade', 'total_stats']
scaler = MinMaxScaler()
df2[num_cols] = scaler.fit_transform(df2[num_cols])

# 3) encoding dos tipos (One-Hot)
ohe = OneHotEncoder(sparse_output=False, dtype=int, handle_unknown='ignore')
tipos = ohe.fit_transform(df2[['tipo_principal', 'tipo_secundario']])
df_tipos = pd.DataFrame(tipos, columns=ohe.get_feature_names_out(['tipo_principal', 'tipo_secundario']), index=df2.index)

df_estruturado = pd.concat([df2[['id', 'nome']].reset_index(drop=True),
                            df2[num_cols].reset_index(drop=True),
                            df_tipos.reset_index(drop=True)], axis=1)

df_estruturado.to_csv('pokemon_estruturado.csv', index=False)
print('salvo pokemon_estruturado.csv')
print(df_estruturado.shape)
df_estruturado.head()